# Learned Paradigms — Cross-Paradigm Synthesis Capstone

**Paper 2 — final notebook in the learned-methods arc.**

This notebook is pure synthesis: no model fits, no new analysis.
It draws headline numbers from existing artifacts (03–08) and assembles
the on-axis leaderboard, the Paper 2 money chart, and the unified mechanism
narrative that ties every learned paradigm together.

The binding constraint across all four paradigms is **weak signal-to-noise**.
Every approach that tries to predict noise loses to methods that diversify
robustly. The ML ensemble wins not by breaking the signal barrier but by
combining imperfect signals with a robust portfolio construction step (MSR).
Everything above the ML bar (2.579) is classical diversification.

## Arc summary

| Notebook | Paradigm | Headline Sharpe (OOS 2023–2026) |
|---|---|---|
| [03](03_ml_strategies.ipynb) | ML ensemble (Lasso/RF/XGB → MSR) | **2.579** ← bar |
| [04](04_dl_strategies.ipynb) / [04b](04b_dl_strategies_cuda.ipynb) | DL predict-then-optimise | 2.386 |
| [05](05_dl_portfolio_construction_exploration.ipynb) | DL direct-weight, 2-asset (SPY+IEF) | 1.240 — off-axis |
| [05b](05b_cross_asset_deep_momentum.ipynb) | DL deep momentum, 29-asset | 0.852 — off-axis |
| [06a](06a_rl_reinforce.ipynb) / [06b](06b_rl_ppo.ipynb) | RL N=29 (REINFORCE + PPO) | ≈2.026 — static collapse |
| [06c](06c_rl_single_asset_dqn.ipynb) | RL single-asset DQN (SPY) | 1.20 — off-axis |
| [07](07_unsupervised_representation_learning.ipynb) / [07b](07b_pca_dislocation_macro_momentum_dashboard.ipynb) | Unsupervised / PCA diagnostic | no perf claim |
| [08](08_llm_bl_views.ipynb) | LLM-augmented BL views | 2.284 (Anthropic) |

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent.resolve()

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

FIG_DIR = ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.rcParams.update({
    "font.family":       "serif",
    "font.size":         10,
    "figure.dpi":        120,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.grid":         True,
    "grid.alpha":        0.2,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

print(f"ROOT: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

## §0 — Load and verify headline numbers from artifacts

All numbers below are read directly from stored artifacts.
The canonical source for the on-axis 29-asset leaderboard is
`data/published/full_comparison_with_rl.csv` (40-strategy dataset published
at the close of Session 4).
LLM-BL numbers come from `results/llm/*_diagnostics.json`.
Off-axis numbers come from their respective `results/` subdirectories.

In [ ]:
# ── On-axis 40-strategy published dataset ──────────────────────────────────
df_pub = pd.read_csv(
    ROOT / "data/published/full_comparison_with_rl.csv", index_col=0
)

def _get(name: str) -> dict:
    row = df_pub.loc[name]
    return {
        "strategy": name,
        "sharpe":   round(row["Sharpe"],   3),
        "ann_ret":  round(row["Ann Ret"],  4),
        "ann_vol":  round(row["Ann Vol"],  4),
        "max_dd":   round(row["Max DD"],   4),
        "family":   row["Family"],
    }

# ── LLM-BL numbers from diagnostics JSON ───────────────────────────────────
def _load_diag(name: str) -> dict:
    with open(ROOT / "results/llm" / f"{name}_diagnostics.json") as f:
        return json.load(f)

diag_anthropic = _load_diag("llm-anthropic")
diag_openai    = _load_diag("llm-openai")
diag_eq        = _load_diag("equilibrium")

# ── Off-axis artifact readers ───────────────────────────────────────────────
df_05  = pd.read_csv(ROOT / "results/notebook_05/comparison.csv")
df_05b = pd.read_csv(ROOT / "results/notebook_05b/comparison.csv")
df_06c = pd.read_csv(ROOT / "results/notebook_06c/summary.csv")

print("Artifacts loaded.")
print(f"  Published dataset : {len(df_pub)} strategies")
print(f"  LLM-Anthropic     : Sharpe={diag_anthropic['sharpe_ratio']:.4f}, refits={diag_anthropic['n_refits']}")
print(f"  LLM-OpenAI        : Sharpe={diag_openai['sharpe_ratio']:.4f}, refits={diag_openai['n_refits']}")
print(f"  05 best DL        : {df_05.iloc[2]['strategy']} Sharpe={df_05.iloc[2]['sharpe']:.4f}")
print(f"  05b DeepMom       : {df_05b.iloc[1]['strategy']} Sharpe={df_05b.iloc[1]['sharpe']:.4f}")
print(f"  06c DQN           : Sharpe={df_06c[df_06c.strategy=='DQN_ensemble']['sharpe'].values[0]:.4f}")

## §1 — On-Axis 29-Asset 2023–2026 Leaderboard

**Comparability condition:** all strategies below share the same evaluation
harness — 29 assets, OOS window 2023-01-01 → 2026-04-30, monthly rebalancing,
net 10 bps one-way transaction costs.  
Strategies trained on 2003-2022 data; 2023+ is genuinely unseen.

The ML bar (**2.579**) is the reference. Nothing clears it; the next-best
is classical VMP(MDP(LW)) at 2.422.

In [ ]:
# On-axis entries — read from artifacts, augmented with LLM-BL
onaxis = [
    # strategy, sharpe, ann_ret, ann_vol, max_dd, paradigm, notebook
    # Source: data/published/full_comparison_with_rl.csv
    ("MSR(Ensemble_μ̂)",       2.579, 16.6, 6.0, -5.9,  "ML",        "03"),
    ("VMP(MDP(LW))",           2.422, 14.9, 5.8, -4.8,  "Classical", "01"),
    ("MSR(RF_μ̂)",              2.394, 20.9, 8.1, -6.8,  "ML",        "03"),
    ("MSR(MLP_μ̂)",             2.386, 23.1, 8.9, -9.3,  "DL",        "04b"),
    ("SignalTilt(XGB)",        2.304, 70.7, 24.5, -22.6, "ML",        "03"),
    # Source: results/llm/llm-anthropic_diagnostics.json (Anthropic = Opus)
    ("BL-LLM(Anthropic/Opus)", round(diag_anthropic["sharpe_ratio"], 3),
     round(diag_anthropic["annualized_return"] * 100, 1),
     round(diag_anthropic["annualized_volatility"] * 100, 1),
     round(diag_anthropic["max_drawdown"] * 100, 1),
     "LLM-BL", "08"),
    # Source: data/published/full_comparison_with_rl.csv
    ("RL(PPO)",                2.027, 22.1, 10.1, -12.2, "RL",        "06b"),
    ("RL(REINFORCE)",          2.026, 22.1, 10.1, -12.2, "RL",        "06a"),
]

cols = ["Strategy", "Sharpe", "Ann Ret (%)", "Ann Vol (%)", "Max DD (%)", "Paradigm", "Notebook"]
leaderboard = pd.DataFrame(onaxis, columns=cols)
leaderboard = leaderboard.sort_values("Sharpe", ascending=False).reset_index(drop=True)

def _fmt_row(r):
    bar = " ← ML bar" if r["Strategy"] == "MSR(Ensemble_μ̂)" else ""
    return [
        r["Strategy"],
        f"{r['Sharpe']:.3f}{bar}",
        f"{r['Ann Ret (%)']:.1f}%",
        f"{r['Ann Vol (%)']:.1f}%",
        f"{r['Max DD (%)']:.1f}%",
        r["Paradigm"],
        r["Notebook"],
    ]

display_df = pd.DataFrame(
    [_fmt_row(r) for _, r in leaderboard.iterrows()],
    columns=cols,
)
display_df.index = range(1, len(display_df) + 1)
display(display_df)

print("\nVerification:")
print(f"  ML bar            : {leaderboard[leaderboard.Paradigm=='ML'].Sharpe.max():.4f}")
print(f"  Best classical    : {leaderboard[leaderboard.Paradigm=='Classical'].Sharpe.max():.4f}")
print(f"  Best DL (on-axis) : {leaderboard[leaderboard.Paradigm=='DL'].Sharpe.max():.4f}")
print(f"  LLM-BL            : {leaderboard[leaderboard.Paradigm=='LLM-BL'].Sharpe.values[0]:.4f}")
print(f"  Best RL           : {leaderboard[leaderboard.Paradigm=='RL'].Sharpe.max():.4f}")

## §2 — Off-Axis Results

**These results are NOT comparable to §1.** Each tests a paradigm-extension
question on a different setup (2-asset universe, different test window, or
single-asset problem). They are reported as confirmatory evidence, not as
leaderboard entries.

| Notebook | Setup | Best DL/RL Sharpe | Reference Sharpe | Reference | Verdict |
|---|---|---|---|---|---|
| 05 | DL direct-weight, 2-asset (SPY+IEF), 2023–2026 | 1.240 | 1.247 | Risk Parity | gap −0.007 — DL ties RP |
| 05b | Cross-asset deep momentum, 29-asset, 2015–2026 | 0.852 | 0.989 | Risk Parity | gap −0.137 — DL loses |
| 06c | Single-asset DQN, SPY, 2-action, 2023–2026 | 1.203 | 1.331 | Buy & Hold | gap −0.128 — RL loses; no timing skill |
| 07/07b | PCA + clustering diagnostic, 29-asset | — | — | — | structural description only |

All four reach the same conclusion: the weak signal-to-noise binding constraint
holds even in simplified setups. Off-axis tests confirm rather than contradict §1.

In [ ]:
# Verify off-axis numbers directly from artifacts

# Notebook 05 — best DL vs best benchmark
_05_best_dl = df_05[df_05.family.str.startswith("DL")].sort_values("sharpe", ascending=False).iloc[0]
_05_best_bm = df_05[df_05.family == "Benchmark"].sort_values("sharpe", ascending=False).iloc[0]
print(f"05  best DL : {_05_best_dl.strategy} Sharpe={_05_best_dl.sharpe:.4f}")
print(f"05  best BM : {_05_best_bm.strategy} Sharpe={_05_best_bm.sharpe:.4f}")
print()

# Notebook 05b — deep momentum vs risk parity
_05b_dm  = df_05b[df_05b.arch == "DeepMomentum"].sort_values("sharpe", ascending=False).iloc[0]
_05b_rp  = df_05b[df_05b.strategy == "RiskParity-21d"].iloc[0]
print(f"05b best DM : {_05b_dm.strategy} Sharpe={_05b_dm.sharpe:.4f}")
print(f"05b RP ref  : {_05b_rp.strategy} Sharpe={_05b_rp.sharpe:.4f}")
print()

# Notebook 06c — DQN vs buy-and-hold
_06c_dqn = df_06c[df_06c.strategy == "DQN_ensemble"].iloc[0]
_06c_bh  = df_06c[df_06c.strategy == "BuyAndHold_SPY"].iloc[0]
print(f"06c DQN     : Sharpe={_06c_dqn.sharpe:.4f}, hit_rate={_06c_dqn.hit_rate:.3f}")
print(f"06c B&H SPY : Sharpe={_06c_bh.sharpe:.4f}")

## §3 — The Money Chart

Horizontal bar chart of the §1 on-axis leaderboard, paradigm-coloured,
with the ML bar (2.579) as a dashed reference line.
Saved to `docs/figures/09_complexity_doesnt_pay.{png,svg}`.

In [ ]:
PARADIGM_COLORS = {
    "ML":        "#1f77b4",   # blue
    "DL":        "#2ca02c",   # green
    "RL":        "#d62728",   # red
    "LLM-BL":   "#9467bd",   # purple
    "Classical": "#7f7f7f",   # grey
}

ML_BAR = 2.579

# Sort descending for display (bottom = worst)
lb = leaderboard.sort_values("Sharpe", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5.5))

y    = np.arange(len(lb))
bars = ax.barh(
    y,
    lb["Sharpe"],
    color=[PARADIGM_COLORS[p] for p in lb["Paradigm"]],
    height=0.65,
    edgecolor="white",
    linewidth=0.4,
)

# ML bar reference line
ax.axvline(ML_BAR, color="#1f77b4", linestyle="--", linewidth=1.4,
           label=f"ML bar  {ML_BAR:.3f}")

# Sharpe labels
for bar, sharpe in zip(bars, lb["Sharpe"]):
    ax.text(
        sharpe + 0.012, bar.get_y() + bar.get_height() / 2,
        f"{sharpe:.3f}",
        va="center", ha="left", fontsize=8.5,
    )

ax.set_yticks(y)
ax.set_yticklabels(lb["Strategy"], fontsize=9)
ax.set_xlabel("Sharpe Ratio (OOS 2023–2026, net 10 bps)")
ax.set_xlim(0, 3.05)
ax.set_title(
    "Complexity Doesn't Pay — On-Axis Leaderboard (29 assets, OOS 2023–2026)",
    fontsize=11, fontweight="bold",
)

# Legend
handles = [
    mpatches.Patch(color=c, label=p)
    for p, c in PARADIGM_COLORS.items()
]
handles.append(
    plt.Line2D([0], [0], color="#1f77b4", linestyle="--", linewidth=1.4,
               label=f"ML bar {ML_BAR:.3f}")
)
ax.legend(handles=handles, loc="lower right", fontsize=8.5,
          framealpha=0.85, edgecolor="#cccccc")

fig.tight_layout()

png_path = FIG_DIR / "09_complexity_doesnt_pay.png"
svg_path = FIG_DIR / "09_complexity_doesnt_pay.svg"
fig.savefig(png_path, dpi=150, bbox_inches="tight")
fig.savefig(svg_path, bbox_inches="tight")
plt.show()
print(f"Saved: {png_path.relative_to(ROOT)}")
print(f"Saved: {svg_path.relative_to(ROOT)}")

## §4 — Unified Mechanism

### Weak signal-to-noise is the binding constraint

Every learned paradigm in this study operates on the same 29-asset
cross-asset universe over the same out-of-sample window.  The headline
finding — that no learned approach clears the ML bar, and the ML bar
itself barely clears the best classical strategy — is not a quirk of
implementation. It reflects a structural property of financial return data:
signal-to-noise ratios are extremely low (cross-asset IC ≈ 0.07 at 252-day
horizon, near-zero at 21 days), and the test window is short enough that
sampling variance in realized Sharpe dominates any predictive edge.

**ML (03)** wins not by predicting returns well, but by combining three
imperfect predictions (Lasso / RF / XGBoost) and feeding the ensemble
into MSR — a robust portfolio construction step that amplifies modest
directional signals while diversifying across asset classes.  
The ensemble reduces idiosyncratic model noise; MSR finds the efficient
frontier tangency under the stabilized expected-return vector.

**DL (04/04b)** adds parameters without adding signal. Sequence models
(LSTM, Transformer) capture temporal patterns, but those patterns are
either already captured by momentum features or are noise. MSR(MLP_μ̂)
Sharpe 2.386 — gap of −0.193 to the ML bar — is entirely within one
standard deviation of seed variance (±0.148). DL is not worse than ML;
it is within noise.

**RL (06a/06b)** exhibits static collapse: both REINFORCE and PPO converge
to ≈1/N equal-weight by mid-training on every walk-forward fold, achieving
Sharpe ≈ 2.027 — essentially the equal-weight baseline (EW Sharpe 2.037).
The mechanism is: high-dimensional continuous simplex actions × weak
reward signal × a short in-sample training window produce a loss landscape
that collapses to the entropy-maximising fixed point.  
The 06c single-asset DQN result (Sharpe 1.20 vs B&H 1.33, hit rate 49%)
confirms that even the simplest RL setup is signal-starved; collapse at
N=29 is expected, not surprising.

**LLM-BL (08)** is the closest challenger at 2.284 (Anthropic/Opus).  The
BL prior anchors the allocation to a robust equilibrium; the LLM layer adds
views that tilt the portfolio toward recent momentum (IC ≈ +0.117 vs
momentum-baseline +0.042).  The informational edge is small and not
statistically significant at N ≈ 41 refits — but the BL anchoring means
the downside of noisy views is bounded.  The training-cutoff contamination
test (Fig B, notebook 08) shows no decay at model knowledge cutoffs,
keeping the result provisionally clean.

**Unsupervised (07/07b)** is the structural diagnostic layer.  PCA explains
~52% of variance in PC1 (risk-on/risk-off) and PC2 (equity/bond), consistent
with the literature.  Clustering reveals 4–5 stable regimes.  These findings
describe the signal landscape rather than predict returns — a necessary
foundation but not itself a source of alpha.

### Implication

The upper bound on Sharpe improvement from model complexity is tightly
constrained by the information content of the input features.  Given
IC ≈ 0.07 at the best horizon, the theoretical maximum Sharpe contribution
from a perfect signal is modest.  The ML ensemble, by stabilising the
optimizer inputs rather than improving the raw forecast accuracy, extracts
nearly all available value.  Additional complexity (DL, RL) does not
improve the signal; it adds fitting noise and training instability.

## §5 — Methods / Governance Bridge

The research producing these results was conducted via a
**governance-first agentic workflow** implemented in `src/aiam/research_agents/`:
an approved artifact manifest, schema validators, and bounded
research-assistant agents that produce structured JSON handoffs before any
result is committed to the paper.

This infrastructure is the responsible substrate for learned-method
experiments: every LLM API call is cached and reproducible, every artifact
is validated against a declared schema, and every strategy result is
traceable to the pipeline step that produced it.

The Cheng–Wu (2024) style multi-tool autonomous agent framework
— capable of spinning up research agents that design and backtest
strategies end-to-end — is committed as **Paper 3** (a separate,
dedicated piece).  It is not claimed as a performance result in Paper 2;
it is a capability demonstration built on the governance substrate
described here.

## §6 — Where This Leaves Us

**Simple beats complex — for the right reasons.**

Across four paradigms and eight sub-studies, the answer is consistent:
robust diversification (VMP, MDP, MSR with stabilised inputs) reliably
outperforms or matches learned complexity on this 29-asset, short-OOS
evaluation.  This is not a failure of implementation — it is the correct
empirical outcome given a weak-signal, short-test-window environment.

**Off-axis results confirm rather than contradict.**  The 2-asset DL
direct-weight probe (05), the cross-asset deep momentum study (05b), and
the single-asset DQN (06c) each test a different structural simplification
of the problem.  All three show the same pattern: learned methods reach
parity with or trail simple benchmarks; signal-to-noise is the binding
constraint regardless of setup.

**LLM-BL is the closest challenger.** At 2.284 it sits 0.295 below the ML
bar and 0.138 below the best classical strategy. The gap is directionally
consistent across 41 refits and both providers, suggesting the BL
anchoring structure extracts modest but real informational value from
Opus views. Whether this clears the bar at longer OOS horizons is the
open question for Paper 2's concluding remarks.

**Paper 3 outlook.**  The full multi-tool agentic build
(Researcher → Backtester → Critic agents, `src/aiam/research_agents/`)
is a **capability demonstration** — showing that a governed agentic
framework can design, implement, and validate novel strategies from a
natural-language specification, with full audit trail.  It is not an
alpha system.  Paper 3 evaluates whether that capability translates to
improved Sharpe on a held-out universe, and whether governance controls
are sufficient to prevent p-hacking in a fully autonomous loop.